In [0]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [0]:
# =========================
# Paths
# =========================
pred_path = "/Volumes/workspace/default/ather/predictions/"
ranking_path = "/Volumes/workspace/default/ather/model_ranking.csv"
fig_path = "/Volumes/workspace/default/ather/figures/"

os.makedirs(fig_path, exist_ok=True)

In [0]:
# =========================
# Load Data
# =========================
ranking_df = pd.read_csv(ranking_path)
base_pred_df = pd.read_csv(pred_path + "predictions.csv")
lstm_ens_df = pd.read_csv(pred_path + "lstm_ensemble_predictions.csv")

## **Group 1:** Performance Plots

In [0]:
# ---- 1. RMSE & R2 Comparison ----
plt.figure(figsize=(10, 6))
x = np.arange(len(ranking_df["Model"]))

plt.bar(x - 0.2, ranking_df["RMSE"], width=0.4, label="RMSE")
plt.bar(x + 0.2, ranking_df["R2"], width=0.4, label="R2")

plt.xticks(x, ranking_df["Model"], rotation=45)
plt.ylabel("Metric Value")
plt.title("Model Comparison (RMSE & R2)")
plt.legend()
plt.tight_layout()

plt.savefig(fig_path + "model_comparison_rmse_r2.png", dpi=300)
plt.show()

In [0]:
# ---- 2. Ranking Chart ----
plt.figure(figsize=(10, 6))
sns.barplot(x="RMSE", y="Model", data=ranking_df, order=ranking_df["Model"])

plt.title("Model Ranking (Lower RMSE = Better)")
plt.xlabel("RMSE")
plt.ylabel("Model")
plt.tight_layout()

plt.savefig(fig_path + "model_ranking.png", dpi=300)
plt.show()

## **Group 2:** Best Model Analysis

In [0]:
best_model = ranking_df.iloc[0]["Model"]

# Select predictions
if best_model in base_pred_df.columns:
    y_true = base_pred_df["y_true"].values
    y_pred = base_pred_df[best_model].values
else:
    y_true = lstm_ens_df["y_true"].values
    y_pred = lstm_ens_df[best_model].values

# Align lengths
min_len = min(len(y_true), len(y_pred))
y_true = y_true[:min_len]
y_pred = y_pred[:min_len]

In [0]:
# ---- 3. Predicted vs Actual ----
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_true, y=y_pred)

plt.xlabel("Actual MSI")
plt.ylabel("Predicted MSI")
plt.title(f"{best_model}: Predicted vs Actual")
plt.tight_layout()

plt.savefig(fig_path + "pred_vs_actual.png", dpi=300)
plt.show()

In [0]:
# ---- 4. Time-Series Plot ----
plt.figure(figsize=(12, 6))
plt.plot(y_true, label="Actual", linewidth=2)
plt.plot(y_pred, label="Predicted", linewidth=2)

plt.xlabel("Time Index")
plt.ylabel("MSI")
plt.title(f"{best_model}: MSI Time-Series Tracking")
plt.legend()
plt.tight_layout()

plt.savefig(fig_path + "time_series_tracking.png", dpi=300)
plt.show()

In [0]:
# ---- 5. Residual Distribution ----
residuals = y_true - y_pred

plt.figure(figsize=(8, 6))
sns.histplot(residuals, bins=50, kde=True)

plt.xlabel("Residual (Actual - Predicted)")
plt.title(f"{best_model}: Residual Distribution")
plt.tight_layout()

plt.savefig(fig_path + "residual_distribution.png", dpi=300)
plt.show()

## **Group 3:** Failure Analysis

In [0]:
# =========================
# Paths
# =========================
pred_path = "/Volumes/workspace/default/ather/predictions/"
ranking_path = "/Volumes/workspace/default/ather/model_ranking.csv"
fig_path = "/Volumes/workspace/default/ather/figures/"

os.makedirs(fig_path, exist_ok=True)

# =========================
# Load Data
# =========================
ranking_df = pd.read_csv(ranking_path)
base_pred_df = pd.read_csv(pred_path + "predictions.csv")
lstm_ens_df = pd.read_csv(pred_path + "lstm_ensemble_predictions.csv")

# Load original dataset (for additional columns like condition, trip_id, temp, etc.)
df = spark.table('workspace.ather.ather_table').toPandas()

# =========================
# Best Model Selection
# =========================
best_model = ranking_df.iloc[0]["Model"]

if best_model in base_pred_df.columns:
    y_true = base_pred_df["y_true"].values
    y_pred = base_pred_df[best_model].values
else:
    y_true = lstm_ens_df["y_true"].values
    y_pred = lstm_ens_df[best_model].values

# Align lengths
min_len = min(len(y_true), len(y_pred), len(df))
y_true = y_true[:min_len]
y_pred = y_pred[:min_len]
df = df.iloc[:min_len]

# =========================
# Compute Error
# =========================
error = y_true - y_pred

In [0]:
# ---- 1. Error vs MSI ----
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_true, y=error)

plt.xlabel("MSI (Actual)")
plt.ylabel("Error")
plt.title(f"{best_model}: Error vs MSI")
plt.tight_layout()

plt.savefig(fig_path + "error_vs_msi.png", dpi=300)
plt.show()


In [0]:
# ---- 2. Error vs Dynamic Stress ----
# Using proxy: Motor_Current__A_
if "Motor_Current__A_" in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=df["Motor_Current__A_"], y=error)

    plt.xlabel("Motor Current (Dynamic Stress Proxy)")
    plt.ylabel("Error")
    plt.title(f"{best_model}: Error vs Dynamic Stress")
    plt.tight_layout()

    plt.savefig(fig_path + "error_vs_dynamic_stress.png", dpi=300)
    plt.show()

In [0]:
# ---- 3. Error vs Temperature ----
if "Motor_Temp__C_" in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=df["Motor_Temp__C_"], y=error)

    plt.xlabel("Motor Temperature (°C)")
    plt.ylabel("Error")
    plt.title(f"{best_model}: Error vs Temperature")
    plt.tight_layout()

    plt.savefig(fig_path + "error_vs_temperature.png", dpi=300)
    plt.show()

## **Group 4:** Robustness Analysis

In [0]:
# ---- 4. MSI across Driving Conditions ----
if "condition" in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df["condition"], y=y_pred)

    plt.xlabel("Driving Condition")
    plt.ylabel("Predicted MSI")
    plt.title(f"{best_model}: MSI across Driving Conditions")
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.savefig(fig_path + "msi_across_conditions.png", dpi=300)
    plt.show()

In [0]:
# ---- 5. MSI across Trips/Drivers ----
if "trip_id" in df.columns:
    # Limit number of trips for readability
    top_trips = df["trip_id"].value_counts().index[:10]
    df_subset = df[df["trip_id"].isin(top_trips)]

    y_pred_subset = y_pred[df_subset.index]

    plt.figure(figsize=(12, 6))
    sns.boxplot(x=df_subset["trip_id"], y=y_pred_subset)

    plt.xlabel("Trip ID")
    plt.ylabel("Predicted MSI")
    plt.title(f"{best_model}: MSI across Trips/Drivers")
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.savefig(fig_path + "msi_across_trips.png", dpi=300)
    plt.show()

## **Group 5:** 

In [0]:
# =========================
# Paths
# =========================
model_path = "/Volumes/workspace/default/ather/models/"
fig_path = "/Volumes/workspace/default/ather/figures/"
data_path = "/Volumes/workspace/default/ather/"
table_path = "/Volumes/workspace/default/ather/"

os.makedirs(fig_path, exist_ok=True)

# =========================
# Load Random Forest Model
# =========================
rf_model = joblib.load(model_path + "random_forest.pkl")

In [0]:
# =========================
# Load Original Data to get Feature Names
# =========================
df = spark.table('workspace.ather.ather_table').toPandas()

target_column = "Thermal_Load_Index"
drop_cols = ["TIMESTAMP", "Timestamp_1"]

df = df.drop(columns=drop_cols)

feature_cols = [c for c in df.columns if c != target_column]

numeric_cols = [
    c for c, t in df.dtypes.items()
    if c in feature_cols and str(t) in ["int64", "float64"]
]

In [0]:
# =========================
# Extract Feature Importances
# =========================
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": numeric_cols,
    "Importance": importances
})

# Sort
importance_df = importance_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [0]:
# =========================
# Save Importance Table
# =========================
importance_df.to_csv(table_path + "feature_importance_table.csv", index=False)


In [0]:
# =========================
# Plot Top 20 Features
# =========================
top20 = importance_df.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=top20)

plt.title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()

plt.savefig(fig_path + "top20_feature_importance.png", dpi=300)
plt.show()


In [0]:
# =========================
# Domain Mapping Function
# =========================
def map_domain(feature):
    feature = feature.lower()
    
    if any(k in feature for k in ["voltage", "current", "power", "dc_link"]):
        return "Electrical"
    elif any(k in feature for k in ["temp", "thermal"]):
        return "Thermal"
    elif any(k in feature for k in ["torque", "speed", "acceleration", "load"]):
        return "Mechanical"
    elif any(k in feature for k in ["soc", "soh", "soe", "cell"]):
        return "Battery"
    else:
        return "Other"

importance_df["Domain"] = importance_df["Feature"].apply(map_domain)


In [0]:
# =========================
# Domain Contribution
# =========================
domain_contribution = importance_df.groupby("Domain")["Importance"].sum().reset_index()
domain_contribution = domain_contribution.sort_values(by="Importance", ascending=False)

# Save Domain Table
domain_contribution.to_csv(table_path + "domain_contribution.csv", index=False)


In [0]:
# =========================
# Plot Domain Contribution
# =========================
plt.figure(figsize=(8, 6))
sns.barplot(x="Domain", y="Importance", data=domain_contribution)

plt.title("Domain Contribution to MSI (Random Forest)")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(fig_path + "domain_contribution.png", dpi=300)
plt.show()